In [19]:
from pathlib import Path
import pandas as pd
from catboost import CatBoostClassifier


In [20]:
#Пути к данным
ROOT = Path(".")
DATA_DIR = ROOT / "data"          
RANDOM_STATE = 42


In [21]:
#Загрузка данных

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print(f"train: {train.shape}")
print(f"test:  {test.shape}")
print(f"target mean: {train['target'].mean():.4f}")

train: (13694, 119)
test:  (4306, 118)
target mean: 0.2075


In [22]:
#Выбор признаков
feature_columns = [
    col for col in train.columns 
    if col not in ['lead_id', 'user_id', 'assignment_ts', 'assignment_date', 'target']]

print(f"total features: {len(feature_columns)}")

total features: 114


In [ ]:
train['assignment_date'] = pd.to_datetime(train['assignment_date']).dt.date
dates = sorted(train['assignment_date'].unique())
cutoff = dates[int(len(dates) * 0.8)]   

train_part = train[train['assignment_date'] < cutoff].copy()
valid_part = train[train['assignment_date'] >= cutoff].copy()

print(f"train part: {len(train_part)} ")
print(f"valid part: {len(valid_part)} ")


# Фиксим категориальные признаки 
categorical_features = [col for col in feature_columns 
                        if train[col].dtype == 'object' or 
                        (train[col].dtype != 'float64' and train[col].nunique() < 30)]

print(f"categorical_features: {len(categorical_features)}")

train part: 10272 
valid part: 3422 
categorical_features: 7


In [24]:
#МОДЕЛЬ CatBoost

model = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.08,
    depth=8,
    eval_metric='PRAUC',
    random_seed=RANDOM_STATE,
    verbose=100,
    early_stopping_rounds=80,
    class_weights=[1, 12]
)

model.fit(
    train_part[feature_columns],
    train_part['target'],
    eval_set=(valid_part[feature_columns], valid_part['target']),
    cat_features=categorical_features,
    use_best_model=True
)

0:	learn: 0.8490978	test: 0.8417622	best: 0.8417622 (0)	total: 89.6ms	remaining: 1m 47s
100:	learn: 0.9509750	test: 0.8917154	best: 0.8917154 (100)	total: 11.6s	remaining: 2m 6s
200:	learn: 0.9904666	test: 0.9026332	best: 0.9030935 (191)	total: 23.2s	remaining: 1m 55s
300:	learn: 0.9982650	test: 0.9052193	best: 0.9056386 (285)	total: 54.5s	remaining: 2m 42s
400:	learn: 0.9997746	test: 0.9060092	best: 0.9068931 (374)	total: 1m 23s	remaining: 2m 45s
500:	learn: 0.9999734	test: 0.9072513	best: 0.9073487 (454)	total: 1m 37s	remaining: 2m 16s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.9078140258
bestIteration = 519

Shrink model to first 520 iterations.


CatBoostClassifier(class_weights=[1, 12], depth=8, early_stopping_rounds=80, eval_metric='PRAUC', iterations=1200, learning_rate=0.08, random_seed=42, verbose=100)

In [25]:
#Создаём submission

final_model = CatBoostClassifier(**model.get_params())
final_model.fit(
    train[feature_columns], 
    train['target'], 
    cat_features=categorical_features
)

test_pred = final_model.predict_proba(test[feature_columns])[:, 1]

submission = pd.DataFrame({
    "lead_id": test["lead_id"].astype(str),
    "score": test_pred
})

submission.to_csv("submission.csv", index=False)

print("Готово!")
print(f"Файл submission.csv создан. Строк: {len(submission)}")
print(submission.head())

0:	learn: 0.8542455	total: 53.8ms	remaining: 1m 4s
100:	learn: 0.9404164	total: 5.74s	remaining: 1m 2s
200:	learn: 0.9826546	total: 11.8s	remaining: 58.8s
300:	learn: 0.9952081	total: 18.1s	remaining: 53.9s
400:	learn: 0.9986972	total: 24.2s	remaining: 48.3s
500:	learn: 0.9996497	total: 30.2s	remaining: 42.1s
600:	learn: 0.9999302	total: 36.5s	remaining: 36.4s
700:	learn: 0.9999861	total: 43s	remaining: 30.6s
800:	learn: 0.9999995	total: 49.3s	remaining: 24.6s
900:	learn: 1.0000000	total: 55.6s	remaining: 18.4s
1000:	learn: 1.0000000	total: 1m 1s	remaining: 12.3s
1100:	learn: 1.0000000	total: 1m 7s	remaining: 6.11s
1199:	learn: 1.0000000	total: 1m 13s	remaining: 0us
Готово!
Файл submission.csv создан. Строк: 4306
                 lead_id     score
0  lead_97e409eb8f8c8246  0.497819
1  lead_55310edb4489f9e9  0.369943
2  lead_e7f653a2c6a7eee8  0.955705
3  lead_22f8e1cfc487ac20  0.044942
4  lead_48b638b839abfac3  0.290289
